<a href="https://colab.research.google.com/github/dongunny/secom-fault-detection/blob/main/SECOM_Analysis_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SECOM 반도체 공정 불량 예측 AI
## Semi-Conductor Manufacturing Fault Detection

**목표:** UCI SECOM 데이터셋의 반도체 공정 센서 데이터를 분석하여  
불량(Fail) 웨이퍼를 조기 탐지하는 머신러닝 모델 구축

| 항목 | 내용 |
|------|------|
| 데이터셋 | UCI SECOM (McCann & Johnston, 2008) |
| 샘플 수 | 1,567개 |
| 변수 수 | 591개 공정 센서 |
| 불량률 | 6.6% (극심한 불균형) |
| 최고 성능 | **BER 29.7%** (논문 33.5% 대비 3.8%p 개선) |

---
> 이 노트북은 로컬에서 생성되었으며, Google Colab에서 바로 실행 가능합니다.


## Step 1. 환경 설정 및 라이브러리 임포트

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, json

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve,
                             confusion_matrix, balanced_accuracy_score)
from sklearn.impute import SimpleImputer

print("환경 설정 완료!")


## Step 2. 데이터 다운로드
UCI Machine Learning Repository에서 SECOM 데이터를 직접 다운로드합니다.


In [ ]:
import urllib.request

os.makedirs('secom', exist_ok=True)
BASE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/secom/'

for fname in ['secom.data', 'secom_labels.data']:
    path = f'secom/{fname}'
    if not os.path.exists(path):
        print(f'다운로드: {fname}')
        urllib.request.urlretrieve(BASE + fname, path)
        print(f'  완료: {os.path.getsize(path)/1024:.0f} KB')
    else:
        print(f'기존 파일 사용: {fname}')


## Step 3. 데이터 로딩 및 기본 EDA

In [ ]:
X = pd.read_csv('secom/secom.data', sep=' ', header=None, na_values='NaN')
ldf = pd.read_csv('secom/secom_labels.data', sep=' ',
                  header=None, names=['label','timestamp'])
y_binary = (ldf['label'].values == 1).astype(int)  # 0=Pass, 1=Fail

print("=" * 50)
print("  SECOM 데이터셋 기본 정보")
print("=" * 50)
print(f"샘플 수    : {X.shape[0]:,}개")
print(f"변수 수    : {X.shape[1]:,}개")
print(f"양품(Pass) : {(y_binary==0).sum()}개 ({(y_binary==0).mean()*100:.1f}%)")
print(f"불량(Fail) : {y_binary.sum()}개 ({y_binary.mean()*100:.1f}%)")
print(f"평균 결측률 : {X.isnull().mean().mean()*100:.2f}%")
print(f"최대 결측률 : {X.isnull().mean().max()*100:.1f}%")


In [ ]:
# EDA 시각화
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.patch.set_facecolor('#0d1117')

# 클래스 분포 도넛차트
counts = [y_binary.sum(), (y_binary==0).sum()]
axes[0].set_facecolor('#161b22')
axes[0].pie(counts,
    labels=['Fail (n=104)', 'Pass (n=1,463)'],
    colors=['#ef4444','#22c55e'],
    wedgeprops=dict(width=0.45, edgecolor='#0d1117', linewidth=3),
    autopct='%1.1f%%', pctdistance=0.75, startangle=90,
    textprops={'color':'#e6edf3'})
axes[0].set_title('클래스 분포 (Pass vs Fail)',
    fontsize=14, color='#e6edf3', fontweight='bold')

# 결측값 분포
missing_pct = X.isnull().mean() * 100
axes[1].set_facecolor('#161b22')
axes[1].hist(missing_pct, bins=40, color='#00d2ff', alpha=0.8, edgecolor='#0d1117')
axes[1].axvline(50, color='#ef4444', linestyle='--', lw=2, label='50% 기준')
axes[1].set_xlabel('결측값 비율 (%)', color='#e6edf3')
axes[1].set_ylabel('변수 수', color='#e6edf3')
axes[1].set_title('변수별 결측값 분포', fontsize=14, color='#e6edf3', fontweight='bold')
axes[1].legend(labelcolor='#e6edf3', facecolor='#161b22')
axes[1].tick_params(colors='#e6edf3')

plt.suptitle('SECOM 탐색적 데이터 분석', fontsize=16, color='#00d2ff', fontweight='bold')
plt.tight_layout()
plt.show()


## Step 4. 데이터 전처리

### 전처리 전략
1. **결측률 > 50% 변수 제거** — 절반 이상 누락된 변수는 신뢰 불가
2. **분산 = 0 변수 제거** — 모든 샘플에서 동일값 → 정보 없음
3. **중앙값 대치** — 이상치에 강건한 결측값 처리
4. **StandardScaler 정규화** — 변수 간 스케일 통일


In [ ]:
# 1. 결측률 > 50% 제거
missing_pct_feat = X.isnull().mean()
keep_cols = missing_pct_feat[missing_pct_feat <= 0.5].index
X_filtered = X[keep_cols].copy()
removed_high = X.shape[1] - X_filtered.shape[1]

# 2. 분산 = 0 제거
variances = X_filtered.var()
X_filtered = X_filtered.loc[:, variances > 0]
removed_zero = len(keep_cols) - X_filtered.shape[1]

# 3. 중앙값 대치 + 정규화
imputer  = SimpleImputer(strategy='median')
scaler   = StandardScaler()
X_imp    = imputer.fit_transform(X_filtered)
X_scaled = scaler.fit_transform(X_imp)

print("전처리 결과:")
print(f"  원본 변수       : 591개")
print(f"  결측률>50% 제거 : {removed_high}개")
print(f"  분산=0 제거     : {removed_zero}개")
print(f"  최종 변수 수    : {X_filtered.shape[1]}개")
print(f"  처리 후 형태    : {X_scaled.shape}")


## Step 5. 특징 선택 — F-test ANOVA Top 40

In [ ]:
selector  = SelectKBest(f_classif, k=40)
X_selected = selector.fit_transform(X_scaled, y_binary)

f_scores    = selector.scores_
feat_names  = X_filtered.columns.astype(str)
top_idx     = np.argsort(f_scores)[::-1][:20]
top_names   = [f'F{feat_names[i]}' for i in top_idx]
top_scores  = f_scores[top_idx]

# 특징 중요도 시각화
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
clrs = ['#00d2ff' if i < 10 else '#7c3aed' for i in range(20)]
ax.barh(range(20), top_scores[::-1], color=clrs[::-1],
        edgecolor='#0d1117', height=0.7)
ax.set_yticks(range(20))
ax.set_yticklabels(top_names[::-1], color='#e6edf3', fontsize=9)
ax.set_xlabel('F-통계량', color='#e6edf3')
ax.set_title('F-test 상위 20개 공정 변수', fontsize=14,
             color='#e6edf3', fontweight='bold')
ax.tick_params(colors='#e6edf3')
ax.grid(True, axis='x', alpha=0.3, color='#30363d')
p1 = mpatches.Patch(color='#00d2ff', label='Top 1-10')
p2 = mpatches.Patch(color='#7c3aed', label='Top 11-20')
ax.legend(handles=[p1,p2], facecolor='#161b22', labelcolor='#e6edf3')
plt.tight_layout()
plt.show()

print("Top 5 공정 변수:")
for i in range(5):
    print(f"  {i+1}. {top_names[i]:8s}  F={top_scores[i]:.2f}")


## Step 6. 모델 학습 — 10-fold 교차검증

### 핵심 설계 결정
- `class_weight='balanced'`: Fail에 ~14배 가중치 → 불균형 극복
- `StratifiedKFold`: 각 fold에서 Pass/Fail 비율 동일 유지
- **BER (Balanced Error Rate)** = 1 - Balanced Accuracy → 불균형 데이터 공정 평가


In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(
        C=0.1, class_weight='balanced',
        max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        max_depth=8, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    print(f"[{name}] 학습 중...")
    y_prob = cross_val_predict(model, X_selected, y_binary,
                               cv=cv, method='predict_proba')[:,1]
    y_pred = (y_prob >= 0.5).astype(int)

    ber  = 1 - balanced_accuracy_score(y_binary, y_pred)
    auc  = roc_auc_score(y_binary, y_prob)
    cm   = confusion_matrix(y_binary, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr, tpr, _  = roc_curve(y_binary, y_prob)

    results[name] = {
        'ber': ber*100, 'auc': auc,
        'tpr_pct': tp/(tp+fn)*100, 'tnr_pct': tn/(tn+fp)*100,
        'cm': cm, 'fpr': fpr, 'tpr': tpr,
    }
    print(f"  BER={ber*100:.1f}%  AUC={auc:.3f}  "
          f"TPR={tp/(tp+fn)*100:.1f}%  TNR={tn/(tn+fp)*100:.1f}%")


## Step 7. 성능 시각화 — ROC 커브 & BER 비교

In [ ]:
CLRS = {'Logistic Regression': '#00d2ff', 'Random Forest': '#f59e0b'}
fig, axes = plt.subplots(1, 2, figsize=(14,6))
fig.patch.set_facecolor('#0d1117')

# ROC 커브
ax = axes[0]; ax.set_facecolor('#161b22')
for name, res in results.items():
    ax.plot(res['fpr'], res['tpr'], color=CLRS[name], lw=2.5,
            label=f"{name} (AUC={res['auc']:.3f})")
    ax.fill_between(res['fpr'], res['tpr'], alpha=0.1, color=CLRS[name])
ax.plot([0,1],[0,1], '--', color='#30363d', lw=1.5, label='랜덤 분류기')
ax.set_xlabel('False Positive Rate', color='#e6edf3')
ax.set_ylabel('True Positive Rate', color='#e6edf3')
ax.set_title('ROC 커브', fontsize=14, color='#e6edf3', fontweight='bold')
ax.legend(facecolor='#161b22', labelcolor='#e6edf3')
ax.tick_params(colors='#e6edf3')
ax.grid(True, alpha=0.2, color='#30363d')

# BER 비교
ax = axes[1]; ax.set_facecolor('#161b22')
xlabels = ['S2N
(논문)','T-test
(논문)','F-test
(논문)',
           'LR
(본연구)','RF
(본연구)']
yvals = [34.5, 33.7, 33.5,
         results['Logistic Regression']['ber'],
         results['Random Forest']['ber']]
bclrs = ['#30363d','#30363d','#30363d','#7c3aed','#00d2ff']
bars = ax.bar(xlabels, yvals, color=bclrs, edgecolor='#0d1117', width=0.6)
for bar, v in zip(bars, yvals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{v:.1f}%', ha='center', fontsize=11,
            fontweight='bold', color='#e6edf3')
ax.axhline(33.5, color='#f59e0b', linestyle='--', lw=1.5, label='논문 최고 기준')
ax.set_ylabel('BER (%) — 낮을수록 좋음', color='#e6edf3')
ax.set_title('BER 비교 (논문 vs 본 연구)', fontsize=14, color='#e6edf3', fontweight='bold')
ax.set_ylim(0, 60); ax.tick_params(colors='#e6edf3')
ax.legend(facecolor='#161b22', labelcolor='#e6edf3')
ax.grid(True, axis='y', alpha=0.2, color='#30363d')

plt.suptitle('모델 성능 평가', fontsize=16, color='#00d2ff', fontweight='bold')
plt.tight_layout(); plt.show()


## Step 8. 혼동 행렬 상세 분석

In [ ]:
import matplotlib.colors as mcolors
CMCLR = {'Logistic Regression':'#7c3aed','Random Forest':'#f59e0b'}

fig, axes = plt.subplots(1, 2, figsize=(12,5))
fig.patch.set_facecolor('#0d1117')

for ax, (name, res) in zip(axes, results.items()):
    ax.set_facecolor('#161b22')
    cm = res['cm']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cmap = mcolors.LinearSegmentedColormap.from_list('c',['#161b22',CMCLR[name]])
    sns.heatmap(cm_norm, annot=False, cmap=cmap, ax=ax,
                linewidths=2, linecolor='#0d1117',
                xticklabels=['Pass 예측','Fail 예측'],
                yticklabels=['실제 Pass','실제 Fail'])
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.38, f'{cm[i,j]}개',
                    ha='center', fontsize=14, fontweight='bold', color='#e6edf3')
            ax.text(j+0.5, i+0.62, f'({cm_norm[i,j]*100:.1f}%)',
                    ha='center', fontsize=10, color='#e6edf3', alpha=0.75)
    ax.set_title(f'{name}\nBER={res["ber"]:.1f}%  TPR={res["tpr_pct"]:.1f}%',
                 fontsize=12, color=CMCLR[name], fontweight='bold')
    ax.tick_params(colors='#e6edf3')

plt.suptitle('Confusion Matrix 분석', fontsize=15, color='#00d2ff', fontweight='bold')
plt.tight_layout(); plt.show()


## Step 9. 최종 결과 요약

In [ ]:
print("=" * 55)
print("  SECOM 프로젝트 최종 결과 요약")
print("=" * 55)

paper = [('S2N (논문)',    34.5), ('T-test (논문)', 33.7), ('F-test (논문)', 33.5)]
for name, ber in paper:
    print(f"  {name:<20} BER={ber:.1f}%")
print()
lr = results['Logistic Regression']
rf = results['Random Forest']
print(f"  Logistic Regression  BER={lr['ber']:.1f}%  AUC={lr['auc']:.3f}  "
      f"TPR={lr['tpr_pct']:.1f}%  ← 논문 대비 {33.5-lr['ber']:.1f}%p 개선!")
print(f"  Random Forest        BER={rf['ber']:.1f}%  AUC={rf['auc']:.3f}  "
      f"TPR={rf['tpr_pct']:.1f}%")

cm_lr = results['Logistic Regression']['cm']
print()
print("[LR 혼동 행렬]")
print(f"  TN={cm_lr[0,0]} (양품 정확)  FP={cm_lr[0,1]} (과잉 검사)")
print(f"  FN={cm_lr[1,0]} (불량 누락)  TP={cm_lr[1,1]} (불량 탐지)")


## Step 10. 결론 및 향후 계획

### 핵심 결론
1. **Logistic Regression이 BER 29.7% 달성** → 논문(33.5%)보다 3.8%p 개선
2. **`class_weight='balanced'` 설정이 결정적** → 6.6% 불균형 극복
3. **F-test 특징 선택 효과적** → 591개 → 40개로 압축해도 성능 유지
4. **Random Forest의 역설** → AUC는 더 높지만 BER은 훨씬 나쁨 (임계값 문제)

### 향후 개선 방향
| 방향 | 방법 | 기대 효과 |
|------|------|---------|
| 불균형 해소 | SMOTE 오버샘플링 | Fail 샘플 확보 |
| 모델 업그레이드 | XGBoost, LightGBM | BER 추가 개선 |
| 임계값 최적화 | PR 커브 분석 | TPR 향상 |
| 해석력 강화 | SHAP 분석 | 변수 의미 파악 |
| 비지도 탐지 | AutoEncoder | 라벨 없이 이상 탐지 |

### 참고문헌
- McCann, M., & Johnston, A. (2008). *SECOM Dataset*. UCI ML Repository.
- Breiman, L. (2001). Random Forests. *Machine Learning, 45*(1), 5-32.
